In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../dataset/train.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [3]:
df.shape

(7613, 5)

In [4]:
df.sample(5)

,id,keyword,location,text,target
2256,3235,deluged,NaN,Businesses are deluged with inroices.|Make you...,0
983,1422,body%20bag,New York,New Ladies Shoulder Tote Handbag Faux Leather ...,0
1506,2174,catastrophic,Inexpressible Island,The Catastrophic Effects of Hiroshima and Naga...,1
3591,5130,fatal,Thane,11-Year-Old Boy Charged With Manslaughter of T...,1
2125,3053,deaths,NaN,#vaxshill 2 deaths from measles complications ...,1


**Check NULL value rows**

In [5]:
columns_list = df.columns

for col in columns_list:
    print(f"Column name: {col} contains {df[col].isnull().sum()} NULL entries.")

Column name: id contains 0 NULL entries.
Column name: keyword contains 61 NULL entries.
Column name: location contains 2533 NULL entries.
Column name: text contains 0 NULL entries.
Column name: target contains 0 NULL entries.


In [6]:
df['keyword'].value_counts()

keyword
fatalities               45
deluge                   42
armageddon               42
sinking                  41
damage                   41
                         ..
forest%20fire            19
epicentre                12
threat                   11
inundation               10
radiation%20emergency     9
Name: count, Length: 221, dtype: int64

In [7]:
df['location'].value_counts()

location
USA                    104
New York                71
United States           50
London                  45
Canada                  29
                      ... 
MontrÌ©al, QuÌ©bec       1
Montreal                 1
ÌÏT: 6.4682,3.18287      1
Live4Heed??              1
Lincoln                  1
Name: count, Length: 3341, dtype: int64

**Conclusion**
<br/>
Delete columns **location**, In my opinion, there is no usage of this column specially when classifying for the *Disaster*.
<br/>
**keyword** may helps, but there are also so rows which contains missing values.
<br/>
For that we'll fill those value using values which are occuring the most (*mode* value).
<br/>
Final deletion, **id** column, totally crap no use at all.

In [8]:
new_df = df.drop(columns=['location', 'id'])
new_df.shape

(7613, 3)

In [9]:
KEYWORD_FILLER=new_df['keyword'].mode().values[0]

print("Previously: {}".format(new_df['keyword'].isnull().sum()))
new_df.fillna({'keyword': KEYWORD_FILLER}, inplace=True)
print("Now: {}".format(new_df['keyword'].isnull().sum()))

Previously: 61
Now: 0


In [10]:
for col in new_df.columns:
    print(
        f"Column: {col} has {new_df[col].isnull().sum()} NULL values"
    )

Column: keyword has 0 NULL values
Column: text has 0 NULL values
Column: target has 0 NULL values


In [11]:
new_df.sample(5)

,keyword,text,target
2696,detonation,Detonation into the realistic assets entering ...,0
2689,detonation,Do you want to play a game?\nhttp://t.co/sQFp6...,0
5555,rainstorm,major rainstorm happening! I'm gonna lie down ...,1
2482,desolate,The once desolate valley was transformed into ...,0
758,blew%20up,Max blew tf up ! ?????? shots fired ???? #Catf...,0


**Clean the data**

In [12]:
# import nltk
# nltk.download('popular')

In [13]:
import re
from nltk.tokenize import word_tokenize, sent_tokenize

In [14]:
def remove_punctuation_and_number(text):
    cleaned_text = re.sub(r'[^\w\s]', '', text)
    cleaned_text = re.sub(r'[\n\t\r\f\v\b\a]', '', cleaned_text)
    return cleaned_text

In [15]:
remove_punctuation_and_number("Hii there !!!, 9090_ditto that's me cheetos, here is $20")

'Hii there  9090_ditto thats me cheetos here is 20'

In [16]:
print(word_tokenize("hello world, this is me"))
print(sent_tokenize( "hii, julie hello, ben" ))

['hello', 'world', ',', 'this', 'is', 'me']
['hii, julie hello, ben']


In [17]:
def clean_data(text: str) -> list:

    lower_case_text = text.lower()
    cleaned_text = remove_punctuation_and_number(lower_case_text)
    return cleaned_text

In [18]:
new_df["complete_text"] = new_df['keyword'] + " " + new_df['text']
new_df.sample(1)

,keyword,text,target,complete_text
1898,crushed,Jesus Christ that ball was fucking crushed!! #...,0,crushed Jesus Christ that ball was fucking cru...


In [19]:
new_df.drop(columns=['text', 'keyword'], inplace=True)

In [20]:
new_df

,target,complete_text
0,1,fatalities Our Deeds are the Reason of this #e...
1,1,fatalities Forest fire near La Ronge Sask. Canada
2,1,fatalities All residents asked to 'shelter in ...
3,1,"fatalities 13,000 people receive #wildfires ev..."
4,1,fatalities Just got sent this photo from Ruby ...
...,...,...
7608,1,fatalities Two giant cranes holding a bridge c...
7609,1,fatalities @aria_ahrary @TheTawniest The out o...
7610,1,fatalities M1.94 [01:04 UTC]?5km S of Volcano ...
7611,1,fatalities Police investigating after an e-bik...


In [21]:
new_df['complete_text'] = new_df['complete_text'].apply(lambda x: clean_data(x))

In [22]:
new_df.sample(1)

,target,complete_text
958,0,body20bag louis vuitton monogram sophie limite...


In [23]:
new_df.to_csv("../dataset/updated_train.csv")

**Understanding testing dataset file**

In [24]:
test_file = pd.read_csv("../dataset/test.csv")
test_file.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3263 entries, 0 to 3262
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        3263 non-null   int64 
 1   keyword   3237 non-null   object
 2   location  2158 non-null   object
 3   text      3263 non-null   object
dtypes: int64(1), object(3)
memory usage: 102.1+ KB


In [25]:
test_file.drop(columns=['location', 'id'], inplace=True)
test_file.shape

(3263, 2)

In [26]:
test_file.fillna({
    'keyword': KEYWORD_FILLER
}, inplace=True)

In [28]:
test_file["complete_text"] = test_file['keyword'] + " " + test_file['text']
test_file.drop(columns=['keyword', 'text'], inplace=True)
test_file['complete_text'] = test_file['complete_text'].apply(lambda x: clean_data(x))

In [30]:
test_file.to_csv("../dataset/updated_test.csv", index=False)